# Local Validation Post-Processing + Evaluation

Этот notebook запускает **post-processing** поверх уже готовых validation-артефактов:
- `transcript.json`
- `diarization.json`

и затем считает метрики на validation.

## Что делает notebook
1. Проверяет структуру локального репозитория и наличие артефактов.
2. Проверяет, что в каждом окне есть `audio.wav`, `transcript.json`, `summary.json`, `diarization.json`.
3. Запускает `src.pipeline` в режиме:
   - `--skip-asr`
   - `--skip-diarization`
4. Строит:
   - `utterances.jsonl`
   - `utterances_named.jsonl`
   - `speaker_assignments.json`
   - `predictions.jsonl`
   - `extracted_pairs.xlsx`
   - `gold.xlsx`
5. Считает метрики прямо в notebook.

## Важная логика
Даже в режиме `--skip-asr --skip-diarization` текущий `src.pipeline` всё равно локально **перерезает аудио окна** из validation-фильма через `prepare_audio(...)`. Поэтому для запуска нужен:
- validation-фильм (`.mkv` / `.ac3`)
- `data_val.xlsx`
- `audio_profiles/`
- артефакты `windows/*` с `transcript.json` и `diarization.json`

## Где менять пути
Смотрите параметрическую ячейку ниже.


In [127]:
# -*- coding: utf-8 -*-
from pathlib import Path
import os
import sys
import json

# === Настройка путей ===
# Укажите корень локального репозитория.
# Если notebook лежит внутри репозитория, можно использовать Path.cwd()
REPO_ROOT = Path.cwd().parent

# Переименованная папка артефактов с ASR + diarization
ARTIFACTS_INPUT_ROOT = REPO_ROOT / "artifacts" / "validation_status_svoboden_asr_diarization_colab"

# Куда писать локальный post-processing run
POSTPROCESS_OUTPUT_ROOT = REPO_ROOT / "artifacts" / "validation_status_svoboden_local_postprocess_v11"

# Основные входные данные
WINDOWS_DIR = ARTIFACTS_INPUT_ROOT / "windows"
DIARIZATION_SUMMARY_PATH = ARTIFACTS_INPUT_ROOT / "diarization_run_summary.json"
GOLD_EXCEL = REPO_ROOT / "data" / "raw" / "gold" / "data_val.xlsx"
FIT_INPUT = REPO_ROOT / "data" / "processed" / "gold_dialogues.jsonl"
SAMPLES_DIR = REPO_ROOT / "data" / "raw" / "validation" / "audio_profiles"

# media_input нужен, потому что pipeline повторно режет audio.wav локально
MEDIA_INPUT = REPO_ROOT / "data" / "raw" / "validation" / "status_svoboden.mkv"
if not MEDIA_INPUT.exists():
    fallback_media = REPO_ROOT / "data" / "raw" / "validation" / "status_svoboden.ac3"
    if fallback_media.exists():
        MEDIA_INPUT = fallback_media

VALIDATION_DIR = REPO_ROOT / "data" / "raw" / "validation"

# Выходные файлы
EXTRACTED_PAIRS_OUTPUT = POSTPROCESS_OUTPUT_ROOT / "extracted_pairs.xlsx"
GOLD_OUTPUT = POSTPROCESS_OUTPUT_ROOT / "gold.xlsx"

# Параметры post-processing
UNKNOWN_SPEAKER_NAME = "unknown"
UNKNOWN_SPEAKER_LABEL = "unknown_speaker"
SIMILARITY_THRESHOLD = 0.48
TOP_K_CANDIDATES = 3
MIN_FREQ = 1
MIN_SAMPLE_DURATION_SEC = 0.5
MIN_UTTERANCE_DURATION_SEC = 0.7
MIN_TOTAL_DURATION_SEC = 0.8
MAX_TOTAL_DURATION_SEC = 45.0
MAX_PAUSE_WITHIN_UTTERANCE_SEC = 0.8
MAX_TOTAL_UTTERANCE_DURATION_SEC = 20.0
MAX_NONOVERLAP_ASSIGN_DISTANCE_SEC = 1.0

# Технические параметры
LIMIT = None
DIARIZATION_SEGMENT_MODE = "regular"  # auto | regular | exclusive
INTENT_MODE = "combined"              # rule | ml | combined
ML_MODEL = REPO_ROOT / "data" / "models" / "intent_classifier.joblib"
ML_CONFIDENCE_THRESHOLD = 0.35  # keep all ML additions (matches v6 behavior)

DEVICE = "cpu"         # post-processing идёт локально; diarization/ASR уже готовы
FORCE_RERUN = False    # pipeline сам не имеет force-rerun; здесь оставлено как напоминание

print("REPO_ROOT =", REPO_ROOT)
print("ARTIFACTS_INPUT_ROOT =", ARTIFACTS_INPUT_ROOT)
print("POSTPROCESS_OUTPUT_ROOT =", POSTPROCESS_OUTPUT_ROOT)
print("MEDIA_INPUT =", MEDIA_INPUT)
print("WINDOWS_DIR =", WINDOWS_DIR)


REPO_ROOT = c:\MyFiles\VSCode\Git_Rep\AudioIntent
ARTIFACTS_INPUT_ROOT = c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_asr_diarization_colab
POSTPROCESS_OUTPUT_ROOT = c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_local_postprocess_v11
MEDIA_INPUT = c:\MyFiles\VSCode\Git_Rep\AudioIntent\data\raw\validation\status_svoboden.mkv
WINDOWS_DIR = c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_asr_diarization_colab\windows


## Опционально: установка зависимостей

Если вы запускаете notebook в новом локальном окружении, раскомментируйте следующую ячейку.  
Если у вас уже есть рабочее `venv311`, её можно пропустить.


In [128]:
# Раскомментируйте при необходимости:
# %pip install -U pip
# %pip install pandas openpyxl soundfile resemblyzer numpy scipy tqdm pyyaml torch torchaudio faster-whisper

# ffmpeg должен быть доступен в PATH:
# import os
# print("Current PATH entries with 'ffmpeg':")
# for path in os.environ['PATH'].split(';'):
#     if 'ffmpeg' in path.lower():
#         print(f"  {path}")

In [129]:

# Добавляем репозиторий в sys.path для локальных импортов
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

print("Python:", sys.version)
print("pandas:", pd.__version__)


Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
pandas: 3.0.1


## Проверка структуры и наличия обязательных файлов

In [130]:

required_paths = {
    "REPO_ROOT": REPO_ROOT,
    "ARTIFACTS_INPUT_ROOT": ARTIFACTS_INPUT_ROOT,
    "WINDOWS_DIR": WINDOWS_DIR,
    "DIARIZATION_SUMMARY_PATH": DIARIZATION_SUMMARY_PATH,
    "GOLD_EXCEL": GOLD_EXCEL,
    "FIT_INPUT": FIT_INPUT,
    "SAMPLES_DIR": SAMPLES_DIR,
    "VALIDATION_DIR": VALIDATION_DIR,
    "MEDIA_INPUT": MEDIA_INPUT,
}

missing = []
for name, path in required_paths.items():
    exists = path.exists()
    print(f"{name}: {path} -> {exists}")
    if not exists:
        missing.append((name, path))

if missing:
    raise FileNotFoundError("Не найдены обязательные пути: " + "; ".join(f"{n}={p}" for n, p in missing))


REPO_ROOT: c:\MyFiles\VSCode\Git_Rep\AudioIntent -> True
ARTIFACTS_INPUT_ROOT: c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_asr_diarization_colab -> True
WINDOWS_DIR: c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_asr_diarization_colab\windows -> True
DIARIZATION_SUMMARY_PATH: c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_asr_diarization_colab\diarization_run_summary.json -> True
GOLD_EXCEL: c:\MyFiles\VSCode\Git_Rep\AudioIntent\data\raw\gold\data_val.xlsx -> True
FIT_INPUT: c:\MyFiles\VSCode\Git_Rep\AudioIntent\data\processed\gold_dialogues.jsonl -> True
SAMPLES_DIR: c:\MyFiles\VSCode\Git_Rep\AudioIntent\data\raw\validation\audio_profiles -> True
VALIDATION_DIR: c:\MyFiles\VSCode\Git_Rep\AudioIntent\data\raw\validation -> True
MEDIA_INPUT: c:\MyFiles\VSCode\Git_Rep\AudioIntent\data\raw\validation\status_svoboden.mkv -> True


## Проверка артефактов по окнам

In [131]:

window_dirs = sorted([p for p in WINDOWS_DIR.iterdir() if p.is_dir()])
print("window count:", len(window_dirs))

required_window_files = ["audio.wav", "transcript.json", "summary.json", "diarization.json"]

report = []
for wd in window_dirs:
    row = {"window_id": wd.name}
    for fn in required_window_files:
        row[fn] = (wd / fn).exists()
    report.append(row)

artifacts_df = pd.DataFrame(report)
display(artifacts_df.head())

for fn in required_window_files:
    print(f"{fn} count:", int(artifacts_df[fn].sum()))

bad_windows = artifacts_df[~artifacts_df[required_window_files].all(axis=1)]
print("windows with missing files:", len(bad_windows))
if len(bad_windows):
    display(bad_windows)
    raise RuntimeError("Некоторые окна неполные. Сначала исправьте missing artifacts.")


window count: 28


,window_id,audio.wav,transcript.json,summary.json,diarization.json
0,val_001_21001,True,True,True,True
1,val_002_21002,True,True,True,True
2,val_003_21003,True,True,True,True
3,val_004_21005,True,True,True,True
4,val_005_21006,True,True,True,True


audio.wav count: 28
transcript.json count: 28
summary.json count: 28
diarization.json count: 28
windows with missing files: 0


## Краткая sanity-check сводка по diarization

In [132]:

with DIARIZATION_SUMMARY_PATH.open("r", encoding="utf-8") as f:
    diar_summary = json.load(f)

print("num_windows_requested:", diar_summary.get("num_windows_requested"))
print("num_ok:", diar_summary.get("num_ok"))
print("num_skipped_existing:", diar_summary.get("num_skipped_existing"))
print("num_errors:", diar_summary.get("num_errors"))
print("num_missing_audio:", diar_summary.get("num_missing_audio"))

diag_df = pd.DataFrame(diar_summary.get("windows", []))
display(diag_df.head())

if not diag_df.empty:
    print("windows where regular/exclusive speaker count differs:",
          int(diag_df.get("regular_exclusive_speaker_count_differs", pd.Series(dtype=bool)).fillna(False).sum()))
    print("windows where regular/exclusive segment count differs:",
          int(diag_df.get("regular_exclusive_segment_count_differs", pd.Series(dtype=bool)).fillna(False).sum()))

    suspicious = diag_df[
        diag_df.get("regular_exclusive_speaker_count_differs", False)
        | diag_df.get("regular_exclusive_segment_count_differs", False)
    ]
    if len(suspicious):
        print("\nSuspicious windows:")
        display(suspicious[[
            "window_id",
            "num_segments",
            "num_regular_segments",
            "num_exclusive_segments",
            "num_speakers_detected",
            "num_speakers_detected_regular",
            "num_speakers_detected_exclusive",
        ]])


num_windows_requested: 28
num_ok: 27
num_skipped_existing: 1
num_errors: 0
num_missing_audio: 0


,window_id,status,audio_path,transcript_exists,diarization_output,num_segments,num_regular_segments,num_exclusive_segments,regular_exclusive_segment_count_differs,selected_segments_mode,num_speakers_detected,num_speakers_detected_regular,num_speakers_detected_exclusive,regular_exclusive_speaker_count_differs,load_auth_mode,load_auth_fallback_used,used_exclusive_diarization,exclusive_fallback_used
0,val_001_21001,skipped_existing,/content/drive/MyDrive/AudioIntent/data/artifa...,True,/content/drive/MyDrive/AudioIntent/data/artifa...,4,4,4,False,exclusive,2,2,2,False,token,False,True,False
1,val_002_21002,ok,/content/drive/MyDrive/AudioIntent/data/artifa...,True,/content/drive/MyDrive/AudioIntent/data/artifa...,7,7,7,False,exclusive,2,2,2,False,token,False,True,False
2,val_003_21003,ok,/content/drive/MyDrive/AudioIntent/data/artifa...,True,/content/drive/MyDrive/AudioIntent/data/artifa...,1,1,1,False,exclusive,1,1,1,False,token,False,True,False
3,val_004_21005,ok,/content/drive/MyDrive/AudioIntent/data/artifa...,True,/content/drive/MyDrive/AudioIntent/data/artifa...,5,5,5,False,exclusive,2,2,2,False,token,False,True,False
4,val_005_21006,ok,/content/drive/MyDrive/AudioIntent/data/artifa...,True,/content/drive/MyDrive/AudioIntent/data/artifa...,5,6,5,True,exclusive,1,2,1,True,token,False,True,False


windows where regular/exclusive speaker count differs: 5
windows where regular/exclusive segment count differs: 6

Suspicious windows:


,window_id,num_segments,num_regular_segments,num_exclusive_segments,num_speakers_detected,num_speakers_detected_regular,num_speakers_detected_exclusive
4,val_005_21006,5,6,5,1,2,1
5,val_006_21007,1,3,1,1,2,1
9,val_010_21013,2,5,2,1,2,1
11,val_012_21016,1,2,1,1,2,1
13,val_014_21019,2,4,2,1,2,1
14,val_015_21020,5,8,5,2,2,2


## Сборка команды post-processing

Мы используем существующий `src.pipeline` в режиме:
- `--skip-asr`
- `--skip-diarization`

и подсовываем ему уже готовые:
- `--transcript-input-dir`
- `--diarization-input-dir`


In [133]:

import subprocess

POSTPROCESS_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable,
    "-m", "src.pipeline",
    "--gold-excel", str(GOLD_EXCEL),
    "--validation-sheet", "Вал - Статус свободен",
    "--fit-input", str(FIT_INPUT),
    "--validation-dir", str(VALIDATION_DIR),
    "--media-input", str(MEDIA_INPUT),
    "--samples-dir", str(SAMPLES_DIR),
    "--output-dir", str(POSTPROCESS_OUTPUT_ROOT),
    "--extracted-pairs-output", str(EXTRACTED_PAIRS_OUTPUT),
    "--gold-output", str(GOLD_OUTPUT),
    "--transcript-input-dir", str(WINDOWS_DIR),
    "--diarization-input-dir", str(WINDOWS_DIR),
    "--skip-asr",
    "--skip-diarization",
    "--device", DEVICE,
    "--min-freq", str(MIN_FREQ),
    "--min-sample-duration-sec", str(MIN_SAMPLE_DURATION_SEC),
    "--min-utterance-duration-sec", str(MIN_UTTERANCE_DURATION_SEC),
    "--min-total-duration-sec", str(MIN_TOTAL_DURATION_SEC),
    "--max-total-duration-sec", str(MAX_TOTAL_DURATION_SEC),
    "--max-pause-within-utterance-sec", str(MAX_PAUSE_WITHIN_UTTERANCE_SEC),
    "--max-total-utterance-duration-sec", str(MAX_TOTAL_UTTERANCE_DURATION_SEC),
    "--max-nonoverlap-assign-distance-sec", str(MAX_NONOVERLAP_ASSIGN_DISTANCE_SEC),
    "--similarity-threshold", str(SIMILARITY_THRESHOLD),
    "--top-k-candidates", str(TOP_K_CANDIDATES),
    "--unknown-speaker-label", UNKNOWN_SPEAKER_LABEL,
    "--unknown-speaker-name", UNKNOWN_SPEAKER_NAME,
    "--diarization-segment-mode", str(DIARIZATION_SEGMENT_MODE),
    "--intent-mode", str(INTENT_MODE),
    "--ml-model", str(ML_MODEL),
    "--ml-confidence-threshold", str(ML_CONFIDENCE_THRESHOLD),
]

if LIMIT is not None:
    command += ["--limit", str(LIMIT)]

print("Running command:")
print(" ".join(command))


Running command:
c:\MyFiles\VSCode\Git_Rep\AudioIntent\venv311\Scripts\python.exe -m src.pipeline --gold-excel c:\MyFiles\VSCode\Git_Rep\AudioIntent\data\raw\gold\data_val.xlsx --validation-sheet Вал - Статус свободен --fit-input c:\MyFiles\VSCode\Git_Rep\AudioIntent\data\processed\gold_dialogues.jsonl --validation-dir c:\MyFiles\VSCode\Git_Rep\AudioIntent\data\raw\validation --media-input c:\MyFiles\VSCode\Git_Rep\AudioIntent\data\raw\validation\status_svoboden.mkv --samples-dir c:\MyFiles\VSCode\Git_Rep\AudioIntent\data\raw\validation\audio_profiles --output-dir c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_local_postprocess_v11 --extracted-pairs-output c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_local_postprocess_v11\extracted_pairs.xlsx --gold-output c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_local_postprocess_v11\gold.xlsx --transcript-input-dir c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifac

## Запуск post-processing

In [134]:

env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO_ROOT}{os.pathsep}{env.get('PYTHONPATH', '')}"

result = subprocess.run(
    command,
    cwd=str(REPO_ROOT),
    env=env,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print("\n--- STDOUT ---\n")
print(result.stdout if result.stdout else "<empty>")
print("\n--- STDERR ---\n")
print(result.stderr if result.stderr else "<empty>")

if result.returncode != 0:
    raise RuntimeError("Post-processing failed. See logs above.")


Return code: 0

--- STDOUT ---

Loaded the voice encoder model on cpu in 0.05 seconds.
Validation pipeline Р·Р°РІРµСЂС€С‘РЅ. РћРєРѕРЅ СЃ РѕР±СЂР°Р±РѕС‚РєРѕР№: 28
Gold СЃРѕС…СЂР°РЅС‘РЅ РІ: c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_local_postprocess_v11\gold.xlsx
Predictions СЃРѕС…СЂР°РЅРµРЅС‹ РІ: c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_local_postprocess_v11\extracted_pairs.xlsx


--- STDERR ---

<empty>


## Проверка итоговых файлов

In [135]:

postprocess_summary_path = POSTPROCESS_OUTPUT_ROOT / "run_summary.json"

print("EXTRACTED_PAIRS_OUTPUT exists:", EXTRACTED_PAIRS_OUTPUT.exists(), EXTRACTED_PAIRS_OUTPUT)
print("GOLD_OUTPUT exists:", GOLD_OUTPUT.exists(), GOLD_OUTPUT)
print("run_summary exists:", postprocess_summary_path.exists(), postprocess_summary_path)

if not EXTRACTED_PAIRS_OUTPUT.exists():
    raise FileNotFoundError(f"Не найден {EXTRACTED_PAIRS_OUTPUT}")

if not GOLD_OUTPUT.exists():
    raise FileNotFoundError(f"Не найден {GOLD_OUTPUT}")


EXTRACTED_PAIRS_OUTPUT exists: True c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_local_postprocess_v11\extracted_pairs.xlsx
GOLD_OUTPUT exists: True c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_local_postprocess_v11\gold.xlsx
run_summary exists: True c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_local_postprocess_v11\run_summary.json


## Просмотр prediction Excel

In [136]:

gold_df = pd.read_excel(GOLD_OUTPUT)
pred_df = pd.read_excel(EXTRACTED_PAIRS_OUTPUT)

print("Gold shape:", gold_df.shape)
print("Pred shape:", pred_df.shape)

display(gold_df.head())
display(pred_df.head())


Gold shape: (28, 6)
Pred shape: (28, 6)


,ID,Фильм,Время начала,Время окончания,opening,closing
0,21001,Статус: свободен,00:00:47,00:01:04,Никита - Спасибо,NaN
1,21002,Статус: свободен,00:02:13,00:02:24,Никита - А кто это там такой у нас потрепаный ...,NaN
2,21003,Статус: свободен,00:02:34,00:02:38,NaN,Никита - спасибо
3,21005,Статус: свободен,00:04:50,00:04:58,NaN,Никита - Чао
4,21006,Статус: свободен,00:12:23,00:12:31,Никита - Да; Собеседник - Ник,NaN


,ID,Фильм,Время начала,Время окончания,opening,closing
0,21001,Статус: свободен,00:00:47,00:01:04,Ведущий - А сейчас звезда,NaN
1,21002,Статус: свободен,00:02:13,00:02:24,"Никита - А кто это там такой у нас, потрепанны...",NaN
2,21003,Статус: свободен,00:02:34,00:02:38,Никита - Это был Никита Колесников и Бобе,NaN
3,21005,Статус: свободен,00:04:50,00:04:58,Афина - Слушай; Афина - а давай куда -нибудь з...,"Администратор - Чао, Таня"
4,21006,Статус: свободен,00:12:23,00:12:31,Собеседник - да я не могу пытаюсь покончить с ...,NaN


## Быстрая sanity-check статистика по prediction columns

In [137]:

def nonempty_count(series):
    return int(series.fillna("").astype(str).str.strip().replace({"0": ""}).ne("").sum())

print("non-empty opening predictions:", nonempty_count(pred_df["opening"]))
print("non-empty closing predictions:", nonempty_count(pred_df["closing"]))

def count_multi_pairs(series):
    vals = series.fillna("").astype(str)
    return int(vals.str.contains(";", regex=False).sum())

print("opening cells with multiple pairs:", count_multi_pairs(pred_df["opening"]))
print("closing cells with multiple pairs:", count_multi_pairs(pred_df["closing"]))


non-empty opening predictions: 26
non-empty closing predictions: 14
opening cells with multiple pairs: 10
closing cells with multiple pairs: 2


## Evaluation

Ниже — компактная версия логики из `notebooks/evaluation.ipynb`, но с путями к свежесгенерированным
`gold.xlsx` и `extracted_pairs.xlsx`.


In [138]:

from difflib import SequenceMatcher
import numpy as np

def parse_pairs(df, column, category):
    pairs = set()
    for val in df[column]:
        if pd.isna(val):
            continue
        val = str(val).strip()
        if val == "" or val == "0":
            continue
        for pair_str in val.split(";"):
            pair_str = pair_str.strip().lower()
            if not pair_str:
                continue
            if " - " in pair_str:
                speaker, phrase = pair_str.split(" - ", 1)
                pairs.add((speaker.strip(), phrase.strip(), category))
    return pairs

gold_opening = parse_pairs(gold_df, "opening", "opening")
gold_closing = parse_pairs(gold_df, "closing", "closing")
pred_opening = parse_pairs(pred_df, "opening", "opening")
pred_closing = parse_pairs(pred_df, "closing", "closing")

gold_all = gold_opening | gold_closing
pred_all = pred_opening | pred_closing

def jaccard_index(set1, set2):
    if len(set1) == 0 and len(set2) == 0:
        return 1.0
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    return intersection / union if union > 0 else 0.0

def precision_recall_f1(gold_set, pred_set):
    precision = len(gold_set & pred_set) / len(pred_set) if len(pred_set) > 0 else 0.0
    recall = len(gold_set & pred_set) / len(gold_set) if len(gold_set) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

def phrase_similarity(phrase1, phrase2):
    return SequenceMatcher(None, phrase1, phrase2).ratio()

def greedy_alignment_evaluation(gold_pairs, pred_pairs, include_category=True):
    gold_pairs = sorted(gold_pairs)
    pred_pairs = sorted(pred_pairs)
    used_pred = set()
    alignments = []

    matched_pairs = 0
    exact_matches = 0
    similarity_scores = []

    for gold_idx, gold_item in enumerate(gold_pairs):
        gold_speaker, gold_phrase, gold_category = gold_item
        best_j = None
        best_score = -1.0

        for j, pred_item in enumerate(pred_pairs):
            if j in used_pred:
                continue

            pred_speaker, pred_phrase, pred_category = pred_item

            if include_category and gold_category != pred_category:
                continue
            if gold_speaker != pred_speaker:
                continue

            score = phrase_similarity(gold_phrase, pred_phrase)
            if score > best_score:
                best_score = score
                best_j = j

        if best_j is not None:
            used_pred.add(best_j)
            pred_speaker, pred_phrase, pred_category = pred_pairs[best_j]
            matched_pairs += 1
            exact = gold_phrase == pred_phrase
            if exact:
                exact_matches += 1
            similarity_scores.append(best_score)
            alignments.append({
                "gold": gold_item,
                "pred": pred_pairs[best_j],
                "exact": exact,
                "similarity": round(best_score, 4),
            })
        else:
            alignments.append({
                "gold": gold_item,
                "pred": None,
                "exact": False,
                "similarity": 0.0,
            })

    avg_similarity = float(np.mean(similarity_scores)) if similarity_scores else 0.0

    return {
        "total_gold": len(gold_pairs),
        "total_pred": len(pred_pairs),
        "matched_pairs": matched_pairs,
        "exact_matches": exact_matches,
        "avg_similarity": avg_similarity,
        "alignments": alignments,
    }

def evaluate_set(name, gold_set, pred_set):
    jaccard = jaccard_index(gold_set, pred_set)
    precision, recall, f1 = precision_recall_f1(gold_set, pred_set)
    alignment = greedy_alignment_evaluation(gold_set, pred_set, include_category=True)

    print("=" * 70)
    print(name)
    print("=" * 70)
    print(f"Gold pairs:      {len(gold_set)}")
    print(f"Predicted pairs: {len(pred_set)}")
    print(f"Jaccard:         {jaccard:.4f}")
    print(f"Precision:       {precision:.4f}")
    print(f"Recall:          {recall:.4f}")
    print(f"F1:              {f1:.4f}")
    print(f"Matched pairs:   {alignment['matched_pairs']}")
    print(f"Exact matches:   {alignment['exact_matches']}")
    print(f"Avg similarity:  {alignment['avg_similarity']:.4f}")
    return {
        "name": name,
        "gold_pairs": len(gold_set),
        "pred_pairs": len(pred_set),
        "jaccard": jaccard,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "matched_pairs": alignment["matched_pairs"],
        "exact_matches": alignment["exact_matches"],
        "avg_similarity": alignment["avg_similarity"],
        "alignment": alignment,
    }

results = []
results.append(evaluate_set("ALL", gold_all, pred_all))
results.append(evaluate_set("OPENING", gold_opening, pred_opening))
results.append(evaluate_set("CLOSING", gold_closing, pred_closing))

metrics_df = pd.DataFrame([
    {k: v for k, v in item.items() if k != "alignment"}
    for item in results
])
display(metrics_df)


ALL
Gold pairs:      48
Predicted pairs: 52
Jaccard:         0.1364
Precision:       0.2308
Recall:          0.2500
F1:              0.2400
Matched pairs:   34
Exact matches:   8
Avg similarity:  0.4830
OPENING
Gold pairs:      40
Predicted pairs: 36
Jaccard:         0.1515
Precision:       0.2778
Recall:          0.2500
F1:              0.2632
Matched pairs:   27
Exact matches:   7
Avg similarity:  0.5001
CLOSING
Gold pairs:      8
Predicted pairs: 16
Jaccard:         0.0909
Precision:       0.1250
Recall:          0.2500
F1:              0.1667
Matched pairs:   7
Exact matches:   1
Avg similarity:  0.4169


,name,gold_pairs,pred_pairs,jaccard,precision,recall,f1,matched_pairs,exact_matches,avg_similarity
0,ALL,48,52,0.136364,0.230769,0.25,0.240000,34,8,0.482974
1,OPENING,40,36,0.151515,0.277778,0.25,0.263158,27,7,0.500112
2,CLOSING,8,16,0.090909,0.125000,0.25,0.166667,7,1,0.416870


In [139]:
# === Сохранение метрик в файл для сравнения версий ===
import datetime

metrics_summary = {
    "version": str(POSTPROCESS_OUTPUT_ROOT.name),
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "diarization_segment_mode": DIARIZATION_SEGMENT_MODE,
    "counts": {
        "gold_all":     results[0]["gold_pairs"],
        "pred_all":     results[0]["pred_pairs"],
        "gold_opening": results[1]["gold_pairs"],
        "pred_opening": results[1]["pred_pairs"],
        "gold_closing": results[2]["gold_pairs"],
        "pred_closing": results[2]["pred_pairs"],
    },
    "metrics": {
        "all":     {"precision": results[0]["precision"], "recall": results[0]["recall"], "f1": results[0]["f1"],
                    "matched": results[0]["matched_pairs"], "exact": results[0]["exact_matches"], "avg_sim": results[0]["avg_similarity"]},
        "opening": {"precision": results[1]["precision"], "recall": results[1]["recall"], "f1": results[1]["f1"],
                    "matched": results[1]["matched_pairs"], "exact": results[1]["exact_matches"], "avg_sim": results[1]["avg_similarity"]},
        "closing":  {"precision": results[2]["precision"], "recall": results[2]["recall"], "f1": results[2]["f1"],
                    "matched": results[2]["matched_pairs"], "exact": results[2]["exact_matches"], "avg_sim": results[2]["avg_similarity"]},
    },
}

# --- Сохранение метрик этого прогона ---
metrics_path = POSTPROCESS_OUTPUT_ROOT / "eval_metrics.json"
with metrics_path.open("w", encoding="utf-8") as _f:
    json.dump(metrics_summary, _f, ensure_ascii=False, indent=2)
print("Metrics saved to:", metrics_path)

# --- Обновление общего файла сравнения версий ---
comparison_path = REPO_ROOT / "artifacts" / "eval_comparison.json"
if comparison_path.exists():
    with comparison_path.open("r", encoding="utf-8") as _f:
        comparison = json.load(_f)
else:
    comparison = []

_existing = [i for i, e in enumerate(comparison) if e.get("version") == metrics_summary["version"]]
if _existing:
    comparison[_existing[0]] = metrics_summary
else:
    comparison.append(metrics_summary)

with comparison_path.open("w", encoding="utf-8") as _f:
    json.dump(comparison, _f, ensure_ascii=False, indent=2)
print("Comparison updated:", comparison_path)
print()

# --- Компактная таблица всех версий ---
print("%-50s | %4s | %5s | %7s | %7s | %7s" % ("version", "pred", "exact", "P", "R", "F1"))
print("-" * 90)
for _e in comparison:
    _m = _e.get("metrics", {}).get("all", {})
    _c = _e.get("counts", {})
    _ver    = _e.get("version", "?")[-46:]
    _npred  = _c.get("pred_all", "?")
    _nexact = _m.get("exact", "?")
    _prec   = _m.get("precision")
    _rec    = _m.get("recall")
    _f1     = _m.get("f1")
    _p_str  = ("%7.3f" % _prec)  if _prec  is not None else "      ?"
    _r_str  = ("%7.3f" % _rec)   if _rec   is not None else "      ?"
    _f1_str = ("%7.3f" % _f1)    if _f1    is not None else "      ?"
    print("%-50s | %4s | %5s | %s | %s | %s" % (_ver, _npred, _nexact, _p_str, _r_str, _f1_str))
print()

# --- Список пропущенных gold pairs ---
_alignment_records = []
for _cat_result in results:
    _cat_name = _cat_result["name"]
    for _aln in _cat_result["alignment"]["alignments"]:
        _alignment_records.append({
            "category": _cat_name,
            "gold": list(_aln["gold"]) if _aln["gold"] else None,
            "pred": list(_aln["pred"]) if _aln["pred"] else None,
            "exact": _aln["exact"],
            "similarity": _aln["similarity"],
        })

alignment_path = POSTPROCESS_OUTPUT_ROOT / "eval_alignment.json"
with alignment_path.open("w", encoding="utf-8") as _f:
    json.dump(_alignment_records, _f, ensure_ascii=False, indent=2)
print("Alignment saved to:", alignment_path)

_missed = [r for r in _alignment_records if r["pred"] is None]
print("Missed gold pairs (%d):" % len(_missed))
for _m in _missed:
    _g = _m["gold"]
    print("  [%s] %s - %s" % (_m["category"], _g[0], _g[1]))


Metrics saved to: c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\validation_status_svoboden_local_postprocess_v11\eval_metrics.json
Comparison updated: c:\MyFiles\VSCode\Git_Rep\AudioIntent\artifacts\eval_comparison.json

version                                            | pred | exact |       P |       R |      F1
------------------------------------------------------------------------------------------
исходная                                           |   53 |    12 |   0.226 |   0.250 |   0.238
alidation_status_svoboden_local_postprocess_v1     |   31 |     7 |   0.226 |   0.146 |   0.177
alidation_status_svoboden_local_postprocess_v2     |   28 |     9 |   0.321 |   0.188 |   0.237
alidation_status_svoboden_local_postprocess_v3     |   28 |     5 |   0.321 |   0.188 |   0.237
alidation_status_svoboden_local_postprocess_v4     |   28 |     5 |   0.321 |   0.188 |   0.237
alidation_status_svoboden_local_postprocess_v5     |   44 |     5 |   0.205 |   0.188 |   0.196
alidation_stat

## Просмотр проблемных alignment cases

In [140]:

all_alignment_df = pd.DataFrame(results[0]["alignment"])
display(all_alignment_df.head(30))

missed_gold = all_alignment_df[all_alignment_df["pred"].isna()]
print("Missed gold pairs:", len(missed_gold))
display(missed_gold.head(20))


,total_gold,total_pred,matched_pairs,exact_matches,avg_similarity,alignments
0,48,52,34,8,0.482974,"{'gold': ('администратор', 'привет', 'opening'..."
1,48,52,34,8,0.482974,"{'gold': ('алексей ярцев', 'а вы — никита', 'o..."
2,48,52,34,8,0.482974,"{'gold': ('алексей ярцев', 'буду через полчаса..."
3,48,52,34,8,0.482974,"{'gold': ('алексей ярцев', 'да', 'opening'), '..."
4,48,52,34,8,0.482974,"{'gold': ('алексей ярцев', 'добрый вечер', 'op..."
5,48,52,34,8,0.482974,"{'gold': ('алексей ярцев', 'ну, мы дрались', '..."
6,48,52,34,8,0.482974,"{'gold': ('алексей ярцев', 'привет', 'opening'..."
7,48,52,34,8,0.482974,"{'gold': ('афина', 'алё', 'opening'), 'pred': ..."
8,48,52,34,8,0.482974,"{'gold': ('афина', 'афина', 'opening'), 'pred'..."
9,48,52,34,8,0.482974,"{'gold': ('афина', 'да', 'opening'), 'pred': (..."


KeyError: 'pred'

In [41]:

# The alignment is nested dict with metrics AND the alignments list inside it
print("Keys in results[0]['alignment']:", results[0]["alignment"].keys())

# Get the actual alignments list
alignments_list = results[0]["alignment"]["alignments"]
print(f"\nTotal alignments: {len(alignments_list)}")
print(f"First alignment item: {alignments_list[0]}")

# Create dataframe from the alignments list
all_alignment_df = pd.DataFrame(alignments_list)
print("\nDataFrame columns:", all_alignment_df.columns.tolist())
print("DataFrame shape:", all_alignment_df.shape)
display(all_alignment_df.head(30))

# Find missed predictions
missed_gold = all_alignment_df[all_alignment_df["pred"].isna()]
print(f"\nMissed gold pairs: {len(missed_gold)}")
display(missed_gold.head(20))



Keys in results[0]['alignment']: dict_keys(['total_gold', 'total_pred', 'matched_pairs', 'exact_matches', 'avg_similarity', 'alignments'])

Total alignments: 48
First alignment item: {'gold': ('администратор', 'привет', 'opening'), 'pred': ('администратор', 'привет', 'opening'), 'exact': True, 'similarity': 1.0}

DataFrame columns: ['gold', 'pred', 'exact', 'similarity']
DataFrame shape: (48, 4)


,gold,pred,exact,similarity
0,"(администратор, привет, opening)","(администратор, привет, opening)",True,1.0000
1,"(алексей ярцев, а вы — никита, opening)","(алексей ярцев, желаете что -нибудь, opening)",False,0.3125
2,"(алексей ярцев, буду через полчаса, closing)",None,False,0.0000
3,"(алексей ярцев, да, opening)","(алексей ярцев, добрый вечер, opening)",False,0.1429
4,"(алексей ярцев, добрый вечер, opening)",None,False,0.0000
5,"(алексей ярцев, ну, мы дрались, opening)",None,False,0.0000
6,"(алексей ярцев, привет, opening)",None,False,0.0000
7,"(афина, алё, opening)",None,False,0.0000
8,"(афина, афина, opening)",None,False,0.0000
9,"(афина, да, opening)",None,False,0.0000



Missed gold pairs: 30


,gold,pred,exact,similarity
2,"(алексей ярцев, буду через полчаса, closing)",None,False,0.0
4,"(алексей ярцев, добрый вечер, opening)",None,False,0.0
5,"(алексей ярцев, ну, мы дрались, opening)",None,False,0.0
6,"(алексей ярцев, привет, opening)",None,False,0.0
7,"(афина, алё, opening)",None,False,0.0
8,"(афина, афина, opening)",None,False,0.0
9,"(афина, да, opening)",None,False,0.0
10,"(афина, добрый вечер, opening)",None,False,0.0
11,"(афина, привет, opening)",None,False,0.0
13,"(вадим, желаете что-нибудь, opening)",None,False,0.0


## Полезные Copilot prompts для следующих правок

### 1. Сделать в `src.pipeline` настоящий postprocess-only режим без повторного `prepare_audio`
> Refactor `src/pipeline.py` so that in `--skip-asr --skip-diarization` mode it does not require `media_input` and does not call `prepare_audio(...)` if `audio.wav` already exists in the input artifacts directory. Add a new argument `--audio-input-dir` and make the pipeline reuse `audio.wav`, `transcript.json`, and `diarization.json` from existing window folders.

### 2. Исправить логику summary path в batch diarization runner
> In `src/scripts/run_diarization_batch_colab.py`, fix the summary output path logic so that `diarization_run_summary.json` is written to `output_root` when `output_root` is provided, even if `--write-in-place` is enabled for per-window outputs. Keep per-window `diarization.json` files in-place, but store the batch summary in the dedicated output root.

### 3. Выбор между regular/exclusive downstream
> Update downstream post-processing code so that it can choose between `segments`, `regular_segments`, and `exclusive_segments` from `diarization.json` via a CLI argument like `--diarization-segments-mode {selected,regular,exclusive}`. Keep backward compatibility with existing files where only `segments` exists.

### 4. Отдельный lightweight runner только для local post-processing
> Create a new script `src/scripts/run_local_postprocess.py` that reads existing `windows/*/audio.wav`, `transcript.json`, and `diarization.json` artifacts, builds utterances, speaker assignments, predictions, and `extracted_pairs.xlsx` without running ASR or diarization. Use the same helper functions as `src.pipeline` and avoid code duplication.
